In [2]:
# ============================================================
# 01 — EXPLORACIÓN UTXO (baches + maxlimit)
# BTC QUANT RESEARCH — NOTEBOOK INSTITUCIONAL
# ============================================================

import clickhouse_connect
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# ------------------------------------------------------------
# 1. Conexión a ClickHouse
# ------------------------------------------------------------
client = clickhouse_connect.get_client(host="localhost", port=8123)
print("Conectado a ClickHouse.")

# ------------------------------------------------------------
# 2. Parámetros de carga incremental
# ------------------------------------------------------------
BATCH_SIZE = 200_000      # tamaño de cada bache
MAXLIMIT   = 1_000_000    # límite total de UTXOs a procesar
offset     = 0

utxo_chunks = []

print(f"Procesando hasta {MAXLIMIT:,} UTXOs en baches de {BATCH_SIZE:,}...\n")

# ------------------------------------------------------------
# 3. Loop de baches
# ------------------------------------------------------------
while offset < MAXLIMIT:

    print(f"Cargando batch OFFSET {offset:,} ...")

    query = f"""
    SELECT
        o.txid,
        o.vout,
        o.value,
        o.address,
        o.script_type,
        o.block_height,
        o.block_time
    FROM outputs o
    LEFT JOIN inputs i
        ON o.txid = i.prev_txid AND o.vout = i.prev_vout
    WHERE i.prev_txid IS NULL
    ORDER BY o.block_height
    LIMIT {BATCH_SIZE} OFFSET {offset}
    """

    df = client.query_df(query)

    if df.empty:
        print("No hay más filas. Fin del proceso.")
        break

    df["block_time"] = pd.to_datetime(df["block_time"], unit="s", errors="coerce")
    utxo_chunks.append(df)

    offset += BATCH_SIZE

print("\nBaches cargados:", len(utxo_chunks))

# ------------------------------------------------------------
# 4. Unir todos los baches
# ------------------------------------------------------------
df = pd.concat(utxo_chunks, ignore_index=True)
df = df.sort_values("block_time")

print("Total UTXOs cargados:", len(df))
display(df.head())

# ------------------------------------------------------------
# 5. Feature engineering
# ------------------------------------------------------------
df["value_btc"] = df["value"] / 1e8
df["age_days"] = (pd.Timestamp.utcnow() - df["block_time"]).dt.total_seconds() / 86400

bins = [0,1,7,30,90,180,365,2*365,5*365,10*365,100*365]
labels = ["0–1d","1–7d","7–30d","30–90d","90–180d","180–365d","1–2y","2–5y","5–10y","10y+"]

df["age_bucket"] = pd.cut(df["age_days"], bins=bins, labels=labels, right=False)

# ------------------------------------------------------------
# 6. Histogramas
# ------------------------------------------------------------
sns.histplot(np.log1p(df["value_btc"]), bins=200)
plt.title("Distribución logarítmica del valor UTXO")
plt.show()

sns.histplot(df["age_days"], bins=200)
plt.title("Distribución de antigüedad UTXO")
plt.show()

# ------------------------------------------------------------
# 7. Cohortes
# ------------------------------------------------------------
cohort_counts = df["age_bucket"].value_counts().sort_index()

sns.barplot(x=cohort_counts.index, y=cohort_counts.values)
plt.title("Cohortes UTXO por antigüedad")
plt.xticks(rotation=45)
plt.show()

# ------------------------------------------------------------
# 8. Scatter robusto
# ------------------------------------------------------------
df_scatter = df[["age_days","value_btc"]].dropna()
df_scatter = df_scatter.sample(min(50000, len(df_scatter)), random_state=42)

sns.scatterplot(data=df_scatter, x="age_days", y="value_btc", alpha=0.2, s=10)
plt.yscale("log")
plt.title("Valor UTXO vs Antigüedad (muestra 50k)")
plt.show()

# ------------------------------------------------------------
# 9. Heatmap log-log
# ------------------------------------------------------------
df_kde = df[["age_days","value_btc"]].dropna()
df_kde = df_kde.sample(min(100000, len(df_kde)), random_state=42)

df_kde["log_age"] = np.log1p(df_kde["age_days"])
df_kde["log_value"] = np.log1p(df_kde["value_btc"])

sns.kdeplot(
    x=df_kde["log_age"],
    y=df_kde["log_value"],
    fill=True,
    cmap="magma",
    levels=40,
    thresh=0.02
)
plt.title("Densidad log-log: antigüedad vs valor UTXO")
plt.show()

# ------------------------------------------------------------
# 10. Resumen ejecutivo
# ------------------------------------------------------------
print(f"""
RESUMEN EJECUTIVO — UTXO por baches + maxlimit

• Se procesaron {len(utxo_chunks)} baches de {BATCH_SIZE:,} filas.
• Total UTXOs cargados: {len(df):,}.
• MAXLIMIT configurado en {MAXLIMIT:,}.
• El procesamiento incremental evita saturar RAM y permite escalar a millones de UTXOs.
• La distribución del valor UTXO es log-normal.
• Las cohortes de antigüedad revelan patrones claros de acumulación.
• Este notebook constituye la Fase 1 del pipeline cuant institucional.
""")


Conectado a ClickHouse.
Procesando hasta 1,000,000 UTXOs en baches de 200,000...

Cargando batch OFFSET 0 ...


DatabaseError: Received ClickHouse exception, code: 60, server response: Code: 60. DB::Exception: Unknown table expression identifier 'outputs' in scope SELECT o.txid, o.vout, o.value, o.address, o.script_type, o.block_height, o.block_time FROM outputs AS o LEFT JOIN inputs AS i ON (o.txid = i.prev_txid) AND (o.vout = i.prev_vout) WHERE i.prev_txid IS NULL ORDER BY o.block_height ASC LIMIT 0, 200000. (UNKNOWN_TABLE) (for url http://localhost:8123)